In [ ]:
# ============================================================
# Compare XGBoost / RandomForest / LightGBM
# Leave-One-Firing-Out (LOO by fuel_type)
# Inputs  : X_Value, Force, Pressure_0, Gox
# Outputs : Pressure, rdot_mm_s, Isp, Thrust, cstar
# ============================================================
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [ ]:
# ============================================================
# CONFIG  (keep in sync with your single-model scripts)
# ============================================================

EXCEL_PATH = Path("daq_raw_4blends.xlsx")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

SHEETS_TO_USE = None
REQUIRED_SHEET_COUNT = 4
FUEL_COLUMN = "fuel_type"
FEATURE_COLUMNS = ["X_Value", "Force", "Pressure_0", "Gox"]
TARGET_COLUMNS = ["Pressure", "rdot_mm_s", "Isp", "Thrust", "cstar"]
SEED = 100
MIN_LABELED_ROWS = 4


In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def clean_numeric(df, columns):
    out = df.copy()
    for c in columns:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
        else:
            out[c] = np.nan
    return out


def safe_r2_multi(y_true, y_pred):
    scores = []
    for i in range(y_true.shape[1]):
        if np.unique(y_true[:, i]).size < 2:
            scores.append(np.nan)
        else:
            scores.append(r2_score(y_true[:, i], y_pred[:, i]))
    return np.asarray(scores, dtype=float)

In [ ]:
def load_dataset():
    if not EXCEL_PATH.exists():
        raise FileNotFoundError(EXCEL_PATH.resolve())
    xls = pd.ExcelFile(EXCEL_PATH, engine="openpyxl")
    sheet_names = (xls.sheet_names if SHEETS_TO_USE is None else list(SHEETS_TO_USE))[:REQUIRED_SHEET_COUNT]
    frames = []
    for sheet in sheet_names:
        raw = pd.read_excel(xls, sheet_name=sheet, engine="openpyxl")
        miss = [c for c in FEATURE_COLUMNS if c not in raw.columns]
        if miss:
            raise ValueError(f"Sheet {sheet} missing {miss}")
        keep = list(FEATURE_COLUMNS)
        for t in TARGET_COLUMNS:
            if t not in raw.columns:
                raw[t] = np.nan
            if t not in keep:
                keep.append(t)
        keep = list(dict.fromkeys(keep))
        frame = clean_numeric(raw[keep], keep)
        frame[FUEL_COLUMN] = str(sheet)
        frames.append(frame)
    data = pd.concat(frames, ignore_index=True)
    data = data.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
    if "X_Value" in data.columns:
        data = data[data["X_Value"] >= 0].reset_index(drop=True)
    labeled_mask = data[TARGET_COLUMNS].notna().all(axis=1)
    for t in TARGET_COLUMNS:
        labeled_mask &= np.isfinite(pd.to_numeric(data[t], errors="coerce"))
    n = int(labeled_mask.sum())
    print(f"Sheets: {sheet_names} | Labeled rows: {n}")
    print(f"Validation: Leave-One-Firing-Out by {FUEL_COLUMN}")
    if n < MIN_LABELED_ROWS:
        raise ValueError(f"Need >= {MIN_LABELED_ROWS} labeled rows; found {n}")
    return data.loc[labeled_mask].reset_index(drop=True), sheet_names


def build_pipeline(name, seed, feature_cols):
    pre = ColumnTransformer(
        transformers=[
            ("fuel", make_one_hot_encoder(), [FUEL_COLUMN]),
            ("numeric", "passthrough", feature_cols),
        ],
        remainder="drop",
    )
    if name == "XGBoost":
        base = XGBRegressor(
            n_estimators=300, max_depth=3, learning_rate=0.05, min_child_weight=5,
            reg_alpha=1.0, reg_lambda=5.0, subsample=0.8, colsample_bytree=0.8,
            objective="reg:squarederror", random_state=seed, n_jobs=-1, verbosity=0,
        )
    elif name == "RandomForest":
        base = RandomForestRegressor(
            n_estimators=300, max_depth=12, min_samples_split=5, min_samples_leaf=2,
            max_features="sqrt", random_state=seed, n_jobs=-1,
        )
    else:
        base = LGBMRegressor(
            n_estimators=300, max_depth=3, learning_rate=0.05, num_leaves=8,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=1.0, reg_lambda=5.0, random_state=seed, n_jobs=-1, verbosity=-1,
        )
    return Pipeline([("preprocessor", pre), ("model", MultiOutputRegressor(base))])


def loo_predict(name, labeled, feature_cols):
    X = labeled[feature_cols + [FUEL_COLUMN]]
    y = labeled[TARGET_COLUMNS].to_numpy(dtype=float)
    groups = labeled[FUEL_COLUMN].to_numpy()
    y_hat = np.zeros_like(y, dtype=float)
    logo = LeaveOneGroupOut()
    for fold, (tr, te) in enumerate(logo.split(X, y, groups), start=1):
        pipe = build_pipeline(name, SEED + fold, feature_cols)
        pipe.fit(X.iloc[tr], y[tr])
        y_hat[te] = pipe.predict(X.iloc[te])
    return y_hat


In [ ]:
# ============================================================
# MAIN
# ============================================================

def main():
    labeled, sheet_names = load_dataset()
    feature_cols = list(FEATURE_COLUMNS)
    y_true = labeled[TARGET_COLUMNS].to_numpy(dtype=float)

    rows = []
    print("Models: XGBoost, RandomForest, LightGBM")
    print("-" * 60)
    for name in ["XGBoost", "RandomForest", "LightGBM"]:
        print(f"LOO training {name}...")
        y_hat = loo_predict(name, labeled, feature_cols)
        r2 = safe_r2_multi(y_true, y_hat)
        mae = mean_absolute_error(y_true, y_hat, multioutput="raw_values")
        rmse = np.sqrt(mean_squared_error(y_true, y_hat, multioutput="raw_values"))
        for i, t in enumerate(TARGET_COLUMNS):
            print(f"  {t:<12} LOO R2={r2[i]:7.4f}  MAE={mae[i]:.4f}  RMSE={rmse[i]:.4f}")
            rows.append({
                "model": name, "target": t,
                "loo_r2": float(r2[i]) if np.isfinite(r2[i]) else np.nan,
                "loo_mae": float(mae[i]), "loo_rmse": float(rmse[i]),
            })
        print()

    metrics = pd.DataFrame(rows)
    r2_table = metrics.pivot(index="target", columns="model", values="loo_r2")
    mae_table = metrics.pivot(index="target", columns="model", values="loo_mae")
    rmse_table = metrics.pivot(index="target", columns="model", values="loo_rmse")

    def print_pivot(df, title):
        """Fixed-width pivot table: rows=targets, cols=models."""
        models = list(df.columns)
        col0 = f"{'Target':<14}"
        header = col0 + "".join(f"{m:>14}" for m in models)
        sep = "-" * len(header)
        print(f"\n{title}")
        print(header)
        print(sep)
        for target in df.index:
            line = f"{str(target):<14}"
            for m in models:
                val = df.loc[target, m]
                s = f"{val:.4f}" if np.isfinite(val) else "nan"
                line += f"{s:>14}"
            print(line)
        print(sep)

    print("=" * 60)
    print_pivot(r2_table, "LOO R2 (higher better)")
    print_pivot(mae_table, "LOO MAE (lower better)")
    print_pivot(rmse_table, "LOO RMSE (lower better)")

    best_rows = []
    for t in TARGET_COLUMNS:
        sub = metrics[metrics["target"] == t].sort_values("loo_r2", ascending=False)
        top = sub.iloc[0]
        best_rows.append({
            "target": t, "best_model": top["model"],
            "best_r2": top["loo_r2"], "best_mae": top["loo_mae"], "best_rmse": top["loo_rmse"],
        })
    best_df = pd.DataFrame(best_rows)
    print("\nBEST MODEL PER TARGET (LOO R2)")
    hdr = f"{'Target':<14}{'Best_Model':<16}{'R2':>12}{'MAE':>12}{'RMSE':>12}"
    print(hdr)
    print("-" * len(hdr))
    for _, row in best_df.iterrows():
        r2 = row["best_r2"]
        r2_s = f"{r2:.4f}" if np.isfinite(r2) else "nan"
        print(
            f"{str(row['target']):<14}"
            f"{str(row['best_model']):<16}"
            f"{r2_s:>12}"
            f"{row['best_mae']:>12.4f}"
            f"{row['best_rmse']:>12.4f}"
        )
    print("-" * len(hdr))

    metrics.to_csv(RESULTS_DIR / "model_comparison_LOO.csv", index=False)
    text = [
        "MODEL COMPARISON — Leave-One-Firing-Out (LOO)",
        f"Inputs  : {FEATURE_COLUMNS}",
        f"Outputs : {TARGET_COLUMNS}",
        "",
        "LOO R2", r2_table.round(4).to_string(),
        "", "LOO MAE", mae_table.round(4).to_string(),
        "", "LOO RMSE", rmse_table.round(4).to_string(),
        "", "BEST PER TARGET", best_df.to_string(index=False),
    ]
    (RESULTS_DIR / "model_comparison_LOO.txt").write_text("\n".join(text), encoding="utf-8")
    print(f"\nSaved: {(RESULTS_DIR / 'model_comparison_LOO.csv').resolve()}")
    print(f"Saved: {(RESULTS_DIR / 'model_comparison_LOO.txt').resolve()}")


if __name__ == "__main__":
    main()
